In [1]:
from cirq_sic import *

In [2]:
def is_power_of_two(n: int) -> bool:
    return n > 0 and n & (n - 1) == 0

def get_wh_qubits(d, wh_implementation):
    n = int(np.ceil(np.log2(d)))
    if 2**n == d:
        cols = 2 if wh_implementation == "simple" else 3
        return cirq.GridQubit.rect(cols, n, top=4, left=2)
    else:
        n = int(np.ceil(np.log2(d)) + 1)
        cols = 3 if wh_implementation == "simple" else 4
        return cirq.GridQubit.rect(cols, n, top=4, left=2)

In [3]:
@recirq.json_serializable_dataclass(namespace="recirq.sky_ground", 
                                    registry=recirq.Registry,
                                    frozen=True)
class WHPOVMOnStatesTask:
    dataset_id: str
    processor_id: str
    run_type: str
    qubits: list
    n_shots: int
    optimizer: str

    d: int
    fiducial_description: str
    states_description: str
    wh_implementation: str

    fiducial: Optional[np.array] = None
    fiducial_circuit: Optional[cirq.Circuit] = None
    states: Optional[list] = None
    states_circuits: Optional[list] = None

    @classmethod
    def filename(cls, **specs):
        return (f"{specs['dataset_id']}/"
                f"d{specs['d']}/"
                f"{cls.__name__}/"
                f"{specs['wh_implementation']}/"
                f"{specs['fiducial_description']}/"
                f"{specs['states_description']}/"
                f"{specs['optimizer']}_{specs['run_type']}_n{abbrev_n_shots(specs['n_shots'])}_{specs['processor_id']}_q{abbrev_grid_qubits(specs['qubits'])}")

    @property
    def fn(self):
        return self.__class__.filename(**self.__dict__)
    
    def make_circuits(self):
        if is_power_of_two(self.d):
            n = int(np.log2(self.d))
            if type(self.fiducial_circuit) != type(None):
                prepare_fiducial = self.fiducial_circuit
            else:
                prepare_fiducial = ansatz_circuit(self.fiducial)
            if type(self.states_circuits) != type(None):
                prepare_states = self.states_circuits
            else:
                prepare_states = [ansatz_circuit(state) for state in self.states]

            if self.wh_implementation == "simple":
                state_qubits = self.qubits[:n]
                fiducial_qubits = self.qubits[n:2*n]
                circuits = [cirq.Circuit((prepare_state(state_qubits),\
                                        simple_wh_povm(state_qubits, fiducial_qubits, prepare_fiducial=prepare_fiducial, measure=True)))\
                                            for prepare_state in prepare_states]
            elif self.wh_implementation == "ak":
                ancilla1 = self.qubits[:n]
                ancilla2 = self.qubits[n:2*n]
                state_qubits = self.qubits[2*n:3*n]
                circuits = [cirq.Circuit((prepare_state(state_qubits),\
                                          arthurs_kelly(state_qubits, ancilla1, ancilla2, prepare_fiducial=prepare_fiducial, measure=True)))\
                                            for prepare_state in prepare_states]
        else:
            n = int(np.ceil(np.log2(self.d)) + 1)
            if type(self.fiducial_circuit) != type(None):
                prepare_fiducial = self.fiducial_circuit
            else:
                prepare_fiducial = ansatz_circuit(pad(self.fiducial, 2**n))
            if type(self.states_circuits) != type(None):
                prepare_states = self.states_circuits
            else:
                prepare_states = [ansatz_circuit(pad(state, 2**n)) for state in self.states]

            if self.wh_implementation == "simple":
                state_qubits = self.qubits[:n]
                fiducial_qubits = self.qubits[n:2*n]
                aux_qubits = self.qubits[2*n:2*n+2]
                circuits = [cirq.Circuit((prepare_state(state_qubits),
                                          simple_wh_povm_d(self.d, state_qubits, fiducial_qubits, aux_qubits, prepare_fiducial=prepare_fiducial, measure=True)))\
                                            for prepare_state in prepare_states]
            elif self.wh_implementation == "ak":
                ancilla1_qubits = self.qubits[:n]
                ancilla2_qubits = self.qubits[n:2*n]
                state_qubits = self.qubits[2*n:3*n]
                aux_qubits = self.qubits[3*n:3*n+2]
                circuits = [cirq.Circuit((prepare_state(state_qubits),\
                                          arthurs_kelly_d(self.d, state_qubits, ancilla1_qubits, ancilla2_qubits, aux_qubits, prepare_fiducial=prepare_fiducial, measure=True)))\
                                                    for prepare_state in prepare_states]
        
        return circuits
    
    def process_results(self, results=None, probs=None):
        r = results_to_freqs(results) if type(probs) == type(None) else probs  
        #if not is_power_of_two(self.d):
        #    n = int(np.ceil(np.log2(self.d)) + 1)
        #    r = np.array([mod_d_probabilities(_, self.d, n, 2) for _ in r])
        #if self.wh_implementation == "ak":
        #    r = change_conjugate_convention(r)
        #r = r.T
        return {"r": r}

In [9]:
d = 2
wh_implementation = "ak"
specs = {"dataset_id": "test",
         "processor_id": "willow_pink",
         "run_type": "clean",
         "qubits": get_wh_qubits(d, wh_implementation),
         "n_shots": 50000,
         "optimizer": "cirq",
         "d": d,
         "fiducial": load_sic_fiducial(d),
         "fiducial_description": "numerical_sic",
         "states": [rand_ket(d)],
         "states_description": "rand_ket",
         "wh_implementation": wh_implementation}

In [10]:
base_dir = "data/sky_ground"
task = task_from_specs(WHPOVMOnStatesTask, specs)
run_sky_ground_task(task)

2025-10-28 03:41:08 [INFO] test/d2/WHPOVMOnStatesTask/ak/numerical_sic/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Starting task...
2025-10-28 03:41:08 [INFO] test/d2/WHPOVMOnStatesTask/ak/numerical_sic/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Creating circuits...
2025-10-28 03:41:08 [INFO] test/d2/WHPOVMOnStatesTask/ak/numerical_sic/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Optimizing circuits...
2025-10-28 03:41:08 [INFO] test/d2/WHPOVMOnStatesTask/ak/numerical_sic/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Sampling...
2025-10-28 03:41:08 [INFO] test/d2/WHPOVMOnStatesTask/ak/numerical_sic/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Processing results...
2025-10-28 03:41:08 [INFO] test/d2/WHPOVMOnStatesTask/ak/numerical_sic/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Saving...


In [11]:
np.array(load_results(task)["processed_data"]["r"])

array([[0.172, 0.046, 0.164, 0.618]])

In [12]:
exactify(task)["r"]

array([[0.172, 0.046, 0.165, 0.617]], dtype=float32)

In [13]:
E = wh_povm(load_sic_fiducial(task.d))
np.array([task.states[0].conj() @ e @ task.states[0] for e in E]).real

array([0.122, 0.095, 0.352, 0.431])